In [ ]:
# IMPORTS
import json

import pandas as pd
from emag_reports.logger import logger
from emag_reports.discovery.itemset_mining import find_frequent_itemsets_survival as find_frequent_itemsets
from emag_reports.discovery.survival_explore import build_and_plot_survival_decision_tree
from emag_reports.discovery.survival_explore import plot_kaplan_meier
from emag_reports.discovery.survival_explore import plot_sample_comparison_CoxPH
from emag_reports.discovery.survival_explore import plot_survival_tree_with_kaplan_meier
from emag_reports.discovery.survival_indicators import feature_permutation_importance
from emag_reports.discovery.survival_indicators import score_Brier
from emag_reports.discovery.survival_indicators import score_C_index
from emag_reports.discovery.survival_indicators import score_features_CoxPH
from emag_reports.preprocess import impute_missing_values
from emag_reports.preprocess import preprocess_survival_data
from emag_reports.preprocess import remove_low_quality_attributes
from IPython.display import display
from IPython.display import HTML
from IPython.display import Markdown as md
from itables import show
from mlxtend.frequent_patterns import association_rules
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sksurv.linear_model import CoxPHSurvivalAnalysis
from sksurv.preprocessing import OneHotEncoder

In [ ]:
## DEFAULT PARAMETERS
# DATASET PARAMETERS
TITLE = "Dataset exploration - knowledge discovery (whitebox models)"
NAME = "bone marrow"
CLASS_ATTRIBUTE = json.dumps({"event": "survival_status", "time": "survival_time"})
# DATA PREPARATION
# Unimportant attributes elimination
UNIMPORTANT_ATTRIBUTE_ELIMINATION = True
I = 0.9
S = 0.9
M = 0.9
T = 0.9
# Missing values imputation
MISSING_VALUES_IMPUTATION = True
# Strategy for missing values imputation
STRATEGY_NUMERICAL = 'mean'
STRATEGY_NOMINAL = 'most_frequent'
# MODELS
# Kaplan-Meier (always)
# Cox Proportional Hazard
COX_PROPORTIONAL_HAZARD = True
# Trees
TREES = True
MAX_DEPTH = 5
# Itemset mining and associations
ITEMSETS_AND_ASSOCIATION = True
MAX_K_VALUE = 3
MAX_INTERVALS_VALUE = 10
MIN_CONFIDENCE = 0.5
MIN_SUPP = 0.01

DATASET = None

MAX_ROWS = 500

In [ ]:
# UNPACK VALUES
CLASS_ATTRIBUTE = json.loads(CLASS_ATTRIBUTE)
CLASS_ATTRIBUTE_EVENT, CLASS_ATTRIBUTE_TIME = CLASS_ATTRIBUTE["event"], CLASS_ATTRIBUTE["time"]
# PATCH PARAMETERS WHICH HAVE REQUIRED VALUES FOR SURVIVAL
USE_LOW_CORRELATION_ATTRIBUTE_REMOVAL = False # always False for survival analysis
EXCLUDE_TARGET_COLUMN = True # always True for survival analysis
# SET OTHER CONSTANTS
CONFIDENCE_TYPE = "log-log"
ALPHA = 0.5 # float, regularization parameter for ridge regression penalty

In [ ]:
html_content = f"""
<div style='text-align: center; margin-bottom: 30px; font-family: Arial, sans-serif;'>
    <h1 style='font-size: 2em; color: #2c3e50;'><strong>Knowledge discovery</strong></h1>
    <h2 style='font-size: 1.3em; color: #34495e;'>Extract interpretable patterns from data</h2>
</div>
<div style='margin: 0 auto; width: 90%; font-family: Arial, sans-serif; color: #2c3e50;'>
    <section style='margin-bottom: 30px;'>
        <h2 style='border-bottom: 2px solid #7f8c8d; padding-bottom: 10px;'>Report information</h2>
        <div style='padding-left: 2em; line-height: 1.5;'>
            <p><strong>Report name:</strong> <span style='font-style: italic; color: #2980b9;'>{TITLE}</span></p>
            <p><strong>Dataset name:</strong> <span style='font-style: italic; color: #2980b9;'>{NAME}</span></p>
        </div>
    </section>
    <section style='margin-bottom: 30px;'>
        <h3 style='border-bottom: 1px solid #bdc3c7; padding-bottom: 5px;'>Data preprocessing parameters</h3>
        <div style='padding-left: 2em; line-height: 1.5;'>
            <p>Remove attributes of low quality: <span style='font-style: italic; color: #2980b9;'>{UNIMPORTANT_ATTRIBUTE_ELIMINATION}</span></p>"""

if UNIMPORTANT_ATTRIBUTE_ELIMINATION:
    html_content += f"""
            <div style='padding-left: 2em;'>
                <p>Excessive number of unique values threshold: <span style='font-style: italic; color: #2980b9;'>{I}</span></p>
                <p>Stability threshold: <span style='font-style: italic; color: #2980b9;'>{S}</span></p>
                <p>Missing values threshold: <span style='font-style: italic; color: #2980b9;'>{M}</span></p>
                <p>Text-like - the variable values resemble text threshold: <span style='font-style: italic; color: #2980b9;'>{T}</span></p>
            </div>"""

html_content += f"""
            <p>Replace missing values: <span style='font-style: italic; color: #2980b9;'>{MISSING_VALUES_IMPUTATION}</span></p>"""

if MISSING_VALUES_IMPUTATION:
    html_content += f"""
            <div style='padding-left: 2em;'>
                <p>For numeric attributes by: <span style='font-style: italic; color: #2980b9;'>{STRATEGY_NUMERICAL}</span></p>
                <p>For nominal attributes by: <span style='font-style: italic; color: #2980b9;'>{STRATEGY_NOMINAL}</span></p>
            </div>
        </div>
    </section>"""

if TREES:
    html_content += f"""
    <section style='margin-bottom: 30px;'>
        <h3 style='border-bottom: 1px solid #bdc3c7; padding-bottom: 5px;'>Tree parameters</h3>
        <div style='padding-left: 2em; line-height: 1.5;'>
            <p>Maximum depth: <span style='font-style: italic; color: #2980b9;'>{MAX_DEPTH}</span></p>
        </div>
    </section>
    """

if ITEMSETS_AND_ASSOCIATION:
    html_content += f"""
    <section style='margin-bottom: 30px;'>
        <h3 style='border-bottom: 1px solid #bdc3c7; padding-bottom: 5px;'>Frequent itemsets and association rules parameters</h3>
        <div style='padding-left: 2em; line-height: 1.5;'>
            <p>Exclude the decision attribute (survival time and survival status): <span style='font-style: italic; color: #2980b9;'>{EXCLUDE_TARGET_COLUMN}</span> (always true for survival problems)</p>
            <p>Maximum number of intervals for numeric attributes: <span style='font-style: italic; color: #2980b9;'>{MAX_INTERVALS_VALUE}</span></p>
            <p>Maximum length: <span style='font-style: italic; color: #2980b9;'>{MAX_K_VALUE}</span></p>
            <p>Minimum support: <span style='font-style: italic; color: #2980b9;'>{MIN_SUPP}</span></p>
            <p>Minimum rule confidence: <span style='font-style: italic; color: #2980b9;'>{MIN_CONFIDENCE}</span></p>
        </div>
    </section>
    """

html_content += "</div>"
display(HTML(html_content))

In [ ]:
PREPROCESSING_ERROR = False

In [ ]:
try:
    # Load dataset
    df = pd.read_json(DATASET)
    df.index.name = "index"
    df.columns = [str(col) for col in df.columns]
    df[CLASS_ATTRIBUTE_EVENT] = df[CLASS_ATTRIBUTE_EVENT].astype(int).astype(bool)
    display(md(f"## Dataset sample"))
    show(df.head(25).reset_index(drop=True), show_index=False)
    
    # Preprocess dataset
    display(md(f"## Preprocessed dataset"))
    
    if UNIMPORTANT_ATTRIBUTE_ELIMINATION:
        ISMT_THRESHOLD = {'I': I, 'S': T, 'M': M, 'T': T}
        df,  low_quality_details= remove_low_quality_attributes(df, [CLASS_ATTRIBUTE_EVENT, CLASS_ATTRIBUTE_TIME], ISMT_THRESHOLD)
        display(md("#### Removed Attributes"))
        reasons_combined = {**low_quality_details}
        for reason, details in reasons_combined.items():
            if details is not None and len(details) > 0:
                if isinstance(details, pd.Index):
                    details = details.tolist()  
                details_str = ', '.join(details)
            else:
                details_str = 'None'
            display(md(f"**{reason}:** {details_str}"))
    if MISSING_VALUES_IMPUTATION:
        df = impute_missing_values(df, [CLASS_ATTRIBUTE_EVENT, CLASS_ATTRIBUTE_TIME], STRATEGY_NUMERICAL, STRATEGY_NOMINAL)
    else:
        df = df.dropna()
    
    X_encoded, X_standardized, X, y = preprocess_survival_data(df, CLASS_ATTRIBUTE_EVENT, CLASS_ATTRIBUTE_TIME)
    
    display(md("#### Dataset after preprocessing (sample)"))
    show(df.head(25).reset_index(drop=True), show_index=False)
except Exception as e:
    logger.exception(e)
    PREPROCESSING_ERROR = True
    display(md("An error occurred during preprocessing. No models will be trained."))

In [ ]:
if not PREPROCESSING_ERROR:
    display(md(f"## Kaplan-Meier curve for the whole dataset"))
    try:
        plot_kaplan_meier(df, CLASS_ATTRIBUTE_EVENT, CLASS_ATTRIBUTE_TIME, CONFIDENCE_TYPE)
    except Exception as e:
        logger.exception(e)
        display(md("An error occurred during Kaplan-Meier curve plotting."))

In [ ]:
if COX_PROPORTIONAL_HAZARD and not PREPROCESSING_ERROR:
    display(md("## Cox Proportional Hazards Model"))
    try:
        model_cph = make_pipeline(OneHotEncoder(), StandardScaler(), CoxPHSurvivalAnalysis(alpha=ALPHA))
        model_cph.fit(X_standardized, y)
        c_index = score_C_index(model_cph, X_standardized, y)
        brier_score = score_Brier(model_cph, X_standardized, y, y, CLASS_ATTRIBUTE_TIME)
        display(md(f"**C-index:** {c_index:.4f}"))
        display(md(f"**IBS:** {brier_score:.4f}"))
    
        display(md("#### Cox Proportional Hazards Model - feature importance (C-index) for each feature"))
        feature_importance_CoxPH = score_features_CoxPH(X_standardized, y, ALPHA)
        show(feature_importance_CoxPH)
        display(md("#### Plot of predicted survival functions for selected samples (CoxPH based on the whole dataset)"))
        SAMPLES_COX = X_standardized.sample(5).index
        plot_sample_comparison_CoxPH(model_cph, X_standardized, SAMPLES_COX)
        show(df.loc[SAMPLES_COX].reset_index(drop=True), index=False)
    except Exception as e:
        logger.exception(e)
        display(md("An error occurred during Cox Proportional Hazard model training."))

In [ ]:
if TREES and not PREPROCESSING_ERROR:
    display(md(f"## Survival tree"))
    try:
        clf_surv, clf_decision_paths_df = build_and_plot_survival_decision_tree(
            df, CLASS_ATTRIBUTE_EVENT, CLASS_ATTRIBUTE_TIME, MAX_DEPTH
        )
        c_index = score_C_index(clf_surv, X_encoded.values, y)
        brier_score = score_Brier(clf_surv, X_encoded.values, y, y, CLASS_ATTRIBUTE_TIME)
        display(md(f"**C-index:** {c_index:.4f}"))
        display(md(f"**IBS:** {brier_score:.4f}"))
        display(md("#### Permutation importance - survival tree"))
        feaure_importance = feature_permutation_importance(clf_surv, X_encoded, y)
        show(feaure_importance)
    except Exception as e:
        logger.exception(e)
        display(md("An error occurred during survival tree generation."))

In [ ]:
if TREES and not PREPROCESSING_ERROR:
    display(md("## Survival tree with Kaplan-Meier plots"))
    try:
        plot_survival_tree_with_kaplan_meier(clf_surv, df, X_encoded, clf_decision_paths_df, CLASS_ATTRIBUTE_EVENT, CLASS_ATTRIBUTE_TIME, CONFIDENCE_TYPE)
    except Exception as e:
        logger.exception(e)
        display(md("An error occurred during survival tree with Kaplan-Meier plots generation."))

In [ ]:
if ITEMSETS_AND_ASSOCIATION and not PREPROCESSING_ERROR:
    try:
        frequent_itemsets = find_frequent_itemsets(df, CLASS_ATTRIBUTE_EVENT, CLASS_ATTRIBUTE_TIME, EXCLUDE_TARGET_COLUMN, MIN_SUPP, MAX_K_VALUE, MAX_INTERVALS_VALUE)
    
        display(md("## Frequent itemset mining"))
        if frequent_itemsets.empty:
            display(md("No frequent itemsets found"))
        else:
            display(md("**Frequent itemset mining was conducted on the dataset following one-hot encoding.**"))
            display(md(f"Showing top {MAX_ROWS} or fewer frequent itemsets with the highest support."))
            frequent_itemsets_sample = frequent_itemsets.sort_values(by='support', ascending=False).head(MAX_ROWS)
            show(frequent_itemsets_sample, showIndex=False, maxRows=MAX_ROWS, maxBytes=0)

        display(md("## Associations"))
        if frequent_itemsets.empty:
            display(md("No frequent itemsets found"))
            display(md("Cannot display associations without frequent itemsets"))
        else:
            display(md(f"Showing top {MAX_ROWS} or fewer associations with the highest lift, support, and confidence."))
            rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=MIN_CONFIDENCE)
            filtered_rules = rules[(rules['support'] >= MIN_SUPP) & (rules['confidence'] >= MIN_CONFIDENCE)]
            filtered_rules_sample = filtered_rules.sort_values(['lift', 'support', 'confidence'], ascending=False).head(MAX_ROWS)
            show(filtered_rules_sample[['antecedents', 'consequents', 'lift', 'support', 'confidence']], showIndex=False, maxRows=MAX_ROWS, maxBytes=0)
    except Exception as e:
        logger.exception(e)
        display(md("An error occurred during itemset mining and association rule extraction."))